## Script Summary

This notebook builds a staged entity-normalization pipeline from curated Canavan relationships, validates names with PrimeKG/UMLS, and maps validated entities back to relationship rows.

### Inputs
- Primary input table: `curated_relationships_df`
- Normalized relationship table: `novel_relationships_df`

### Main processing stages
1. Build initial query names from curated relationships.
2. Split names into exact PrimeKG matches and nonmatches.
3. Run UMLS normalization for nonmatched names.
4. Split first-search results into rows with CUI vs without CUI.
5. Run semantic-type quality check for first-search CUI rows.
6. Build second-round suggestion inputs.
7. Generate PrimeKG suggested replacements (SapBERT retrieve + SBERT rerank).
8. Re-check suggested names in UMLS and evaluate type consistency.
9. Build final valid-name pools across stages.
10. Map status/CUI back to curated relationship rows.
11. Add suggested-name mapping back to relationship rows.

### Key intermediate variables
- `query_names` (round 1 and round 2)
- `kg_matched_names`, `kg_nonmatched_names`
- `entity_umls_df`
- `entity_with_cui_after_first_search`, `entity_without_cui_after_first_search`
- `suggested_name_replacement_df`
- `suggested_umls_typecheck_df`
- `valid_names_after_first_search`, `valid_names_after_second_search`, `all_good_names`

### Final outputs used downstream
- `suggested_name_replacement_df`
- `suggested_umls_typecheck_df`
- `relationships_with_status_df`

In [ ]:
# --- Config & Imports ---
# Portable paths: override with env vars CURATION_ROOT, PLUS_KG, UMLS_API_KEY, RELEASE_ROOT.

import os
import pandas as pd
import re
import requests
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from numpy.linalg import norm

_LIT_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
_RELEASE_ROOT = Path(os.environ.get("RELEASE_ROOT", str(_LIT_DIR.parents[1])))
CURATION_ROOT = Path(os.environ.get(
    "CURATION_ROOT",
    str(_RELEASE_ROOT / "dataset" / "PrimeKG-Plus-RD" / "curation_source"),
))
PLUS_KG = Path(os.environ.get("PLUS_KG", str(_RELEASE_ROOT / "dataset" / "PrimeKG-Plus" / "primekg_plus.csv")))
KG_FILE = PLUS_KG
POST_DIR = CURATION_ROOT / "Post curation"
BEFORE_BERT_DIR = POST_DIR / "before_bert"
FINALS_V1_DIR = POST_DIR / "finals_v1"
QC_OUTPUTS_DIR = POST_DIR / "qc_outputs"
INTERMEDIATE_DIR = POST_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
CURATED_CSV = CURATION_ROOT / "Tay-Sachs/Tay-Sachs Disease final.csv"

UMLS_API_KEY = os.environ.get("UMLS_API_KEY", "")
if not UMLS_API_KEY:
    raise ValueError("Set UMLS_API_KEY (NLM UTS API key) before running UMLS search cells.")

SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
SAPBERT_MAX_LEN = 25
SAPBERT_BATCH = 64

TEST_MODE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
pwd

In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████| 103/103 [00:00<00:00, 1988.22it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
ENTITY_TYPE_TO_TUI = {

    # Disease-related concepts
    "disease": [
        "T047",  # Disease or Syndrome (primary disease group)
        "T191",  # Neoplastic Process (tumor/cancer)
        "T048",  # Mental or Behavioral Dysfunction
    ],

    # Gene / protein
    "gene/protein": [
        "T028",  # Gene or Genome
        "T192",  # Receptor (can be treated as protein)
    ],

    # Gene / protein
    "protein/gene": [
        "T028",  # Gene or Genome
        "T192",  # Receptor (can be treated as protein)
    ],

    # Drug / chemical treatment
    "drug": [
        "T121",  # Pharmacologic Substance
        "T200",  # Clinical Drug
        "T195",  # Antibiotic (subset of drug)
    ],

    # Phenotype / symptom / abnormality
    "phenotype": [
        "T033",  # Finding (e.g., hypotonia, weakness)
        "T034",  # Laboratory or Test Result
        "T041",  # Mental Process
        "T184",  # Sign or Symptom
        "T019",  # Congenital Abnormality
        "T020",  # Acquired Abnormality
    ],

    # Biological process (broad)
    "biological_process": [
        "T038",  # Biologic Function
        "T039",  # Physiologic Function
    ],

    # Molecular function (GO-like)
    "molecular_function": [
        "T043",  # Cell Function
    ],

    # Cellular component
    "cellular_component": [
        "",
    ],

    # Pathway (no exact TUI; closest approximation)
    "pathway": [
        "T038",  # Biologic Function
    ],

    # Anatomy
    "anatomy": [
        "T017",  # Anatomical Structure
        "T018",  # Embryonic Structure
        "T021",  # Fully Formed Anatomical Structure
        "T022",  # Body System
        "T023",  # Body Part, Organ, or Organ Component
        "T024",  # Tissue
        "T029",  # Body Location or Region
        "T030",  # Body Space or Junction
    ],

    # Exposure / procedure / measurement
    "exposure": [
        "T060",  # Diagnostic Procedure (e.g., MRI, ultrasound)
        "T061",  # Therapeutic or Preventive Procedure
        "T063",  # Molecular Biology Research Technique
    ],

    # Pathological process
    "pathology": [
        "T046",  # Pathologic Function (e.g., inflammation)
    ],
}

In [16]:
# Load curated relationships
if not CURATED_CSV.exists():
    raise FileNotFoundError(f"Curated CSV not found: {CURATED_CSV}")
curated_relationships_df = pd.read_csv(CURATED_CSV)
curated_relationships_df.columns

Index(['No.', 'Filename', 'DOI/PMID', 'ABSTRACT', 'JOURNAL TYPE ',
       'EXPERIMENT', 'MODEL', 'RELATION', 'DISPLAY_RELATION', 'X_NAME',
       'X_TYPE', 'Y_NAME', 'Y_TYPE', 'NOTE', 'EXPERT OPINION'],
      dtype='str')

In [19]:
# Standardize known column names
curated_relationships_df = curated_relationships_df[['DOI/PMID', 'ABSTRACT', 'JOURNAL TYPE ',
       'EXPERIMENT', 'MODEL', 'RELATION', 'DISPLAY_RELATION', 'X_NAME',
       'X_TYPE', 'Y_NAME', 'Y_TYPE', 'NOTE', 'EXPERT OPINION']]
curated_relationships_df.columns = ['PMID', 'ABIN', 'Journal type', 'experiment',
       'model', 'relation', 'display_relation', 'x_name', 'x_type',
       'y_name', 'y_type', 'note', 'expert opinion']
curated_relationships_df = curated_relationships_df.drop(columns=['display_relation', 'note', 'expert opinion'])
curated_relationships_df.head(3)

,PMID,ABIN,Journal type,experiment,model,relation,x_name,x_type,y_name,y_type
0,10.1186/s13023-025-04030-6,Abstract,original article,in vivo,human,disease_disease,GM2 gangliosidoses,disease,Tay-sachs disease,disease
1,10.1186/s13023-025-04030-6,Abstract,original article,in vivo,human,disease_disease,GM2 gangliosidoses,disease,Sandhoff disease,disease
2,10.1186/s13023-025-04030-6,Abstract,original article,in vivo,human,disease_phenotype_positive,GM1 gangliosidoses,disease,progressive neurodegeneration,phenotype


In [20]:
len(curated_relationships_df)

338

In [21]:
curated_relationships_df.tail(3)

,PMID,ABIN,Journal type,experiment,model,relation,x_name,x_type,y_name,y_type
335,10.1038/s41586-025-09732-2,Introduction (khổ cuối),original article,in vitro,human,disease_protein,Tay-Sachs disease,disease,HEXA nonsense mutation,protein/gene
336,10.1038/s41586-025-09732-2,Introduction (khổ cuối),original article,in vitro,human,drug_effect,prime editing,drug,enzyme activity restoration,phenotype
337,10.1038/s41586-025-09732-2,Introduction (khổ cuối),original article,in vivo,mouse,drug_effect,prime editing,drug,rescued disease pathology,phenotype


In [22]:
curated_relationships_df = curated_relationships_df.dropna()
len(curated_relationships_df)

334

In [23]:
relationship_rows = []
for _, row in curated_relationships_df.iterrows():
    entity1_name = row.get('x_name')
    entity2_name = row.get('y_name')
    if not (isinstance(entity1_name, str) and entity1_name.strip() and isinstance(entity2_name, str) and entity2_name.strip()):
        continue
    relationship_rows.append({
        'entity1': entity1_name.strip(),
        'entity2': entity2_name.strip(),
        'entity_type1': row.get('x_type'),
        'entity_type2': row.get('y_type'),
        'Relation': row.get('relation'),
        'PMID': row.get('PMID'),
    })

novel_relationships_df = pd.DataFrame(relationship_rows)
novel_relationships_df.head(3)


,entity1,entity2,entity_type1,entity_type2,Relation,PMID
0,GM2 gangliosidoses,Tay-sachs disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6
1,GM2 gangliosidoses,Sandhoff disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6
2,GM1 gangliosidoses,progressive neurodegeneration,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6


In [24]:
len(novel_relationships_df)

334

In [25]:
# String exact match against PrimeKG names
# Output lists required by next steps:
# - kg_matched_names
# - kg_nonmatched_names
import re
import unicodedata
def _norm_name(x: str) -> str:
    s = unicodedata.normalize("NFKC", str(x))
    s = s.strip().lower()
    s = s.replace("‑", "-").replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    return s

In [26]:
query_names = []
if "query_names" not in globals() or query_names is None or len(query_names) == 0:
    q1 = novel_relationships_df.get("entity1", pd.Series(dtype=str)).dropna().astype(str)
    q2 = novel_relationships_df.get("entity2", pd.Series(dtype=str)).dropna().astype(str)
    query_names = sorted({
        _norm_name(x)
        for x in pd.concat([q1, q2], ignore_index=True).tolist()
        if _norm_name(x) and _norm_name(x) != "nan"
    })
len(query_names), query_names[:3]

(266,
 ['aav gene therapy',
  'aav-hex gene therapy and bone marrow transplantation combination',
  'aav-php.eb carrying the abe'])

In [27]:
# Build PrimeKG name pool if not already prepared
primekg_entity_names_list = []
if "primekg_entity_names_list" not in globals() or primekg_entity_names_list is None or len(primekg_entity_names_list) == 0:
    kg_df = pd.read_csv(KG_FILE, low_memory=False)
    name_cols = [c for c in kg_df.columns if c.endswith("_name")]
    if not name_cols:
        raise ValueError(f"No columns ending with _name in {KG_FILE}")
    acc = []
    for c in name_cols:
        acc.extend(kg_df[c].dropna().astype(str).tolist())
    # normalize first, then deduplicate
    primekg_entity_names_list = sorted({
        _norm_name(x)
        for x in acc
        if _norm_name(x) and _norm_name(x) != "nan"
    })
primekg_name_norm_set = set(primekg_entity_names_list)
len(primekg_name_norm_set)

127561

In [28]:
kg_matched_names = []
kg_nonmatched_names = []
for q in query_names:
    q_clean = str(q).strip()
    if not q_clean or q_clean.lower() == "nan":
        continue
    if _norm_name(q_clean) in primekg_name_norm_set:
        kg_matched_names.append(q_clean)
    else:
        kg_nonmatched_names.append(q_clean)

In [29]:
# Unique + sorted for deterministic downstream behavior
kg_matched_names = sorted(set(kg_matched_names))
kg_nonmatched_names = sorted(set(kg_nonmatched_names))

# Build dictionaries requested: name -> curated entity_type(s)
name_to_entity_types = {}
for _, r in novel_relationships_df.iterrows():
    n1 = str(r.get("entity1") or "").strip()
    t1 = str(r.get("entity_type1") or "").strip()
    if n1 and n1.lower() != "nan" and t1 and t1.lower() != "nan":
        name_to_entity_types.setdefault(n1, set()).add(t1)

    n2 = str(r.get("entity2") or "").strip()
    t2 = str(r.get("entity_type2") or "").strip()
    if n2 and n2.lower() != "nan" and t2 and t2.lower() != "nan":
        name_to_entity_types.setdefault(n2, set()).add(t2)

In [30]:
name_to_entity_types

{'GM2 gangliosidoses': {'Disease', 'disease'},
 'Tay-sachs disease': {'disease'},
 'Sandhoff disease': {'disease'},
 'GM1 gangliosidoses': {'disease'},
 'progressive neurodegeneration': {'phenotype'},
 'speech difficulties': {'phenotype'},
 'mobility impairment': {'phenotype'},
 'Tay-Sachs disease': {'disease'},
 'Lysosomal storage disorders': {'disease'},
 'central nervous system neurons': {'anatomy'},
 'peripheral nervous system neurons': {'anatomy'},
 'enlarged neuron': {'phenotype'},
 'foamy neuron': {'phenotype'},
 'HEXA': {'disease', 'protein', 'protein/gene'},
 'enzyme activity reduction': {'molecular_function'},
 'c.1495C>T (p.Arg499Cys)': {'mutation'},
 'neuronal vacuolization': {'phenotype'},
 'lysosomal undegraded metabolites accumulation': {'phenotype'},
 'enzyme β-hexosaminidase deficiency': {'phenotype'},
 'scAAV9-HEXM': {'drug'},
 'immune response': {'phenotype'},
 'GM2 catabolism': {'biological_process'},
 'GM2 accumulation': {'phenotype'},
 'N-acetylglucosamine-thiazol

In [31]:
kg_matched_entities = {
    n: " | ".join(sorted(name_to_entity_types.get(n, set())))
    for n in kg_matched_names
}
kg_nonmatched_entities = {
    n: " | ".join(sorted(name_to_entity_types.get(n, set())))
    for n in kg_nonmatched_names
}

print(f"query_names total: {len(query_names)}")
print(f"kg_matched_names: {len(kg_matched_names)}")
print(f"kg_nonmatched_names: {len(kg_nonmatched_names)}")

routing_preview_df = pd.DataFrame(
    {
        "bucket": ["kg_matched_names", "kg_nonmatched_names"],
        "count": [len(kg_matched_names), len(kg_nonmatched_names)],
    }
)
routing_preview_df

query_names total: 266
kg_matched_names: 44
kg_nonmatched_names: 222


,bucket,count
0,kg_matched_names,44
1,kg_nonmatched_names,222


In [32]:
# UMLS search helpers (restored)
UMLS_BASE = 'https://uts-ws.nlm.nih.gov/rest'


def _umls_search_best(term, api_key, max_results=10):
    """Return first valid CUI match from UMLS search (no type constraint)."""
    try:
        import re

        url = f"{UMLS_BASE}/search/current"
        params = {
            'apiKey': api_key,
            'string': term,
            'pageSize': max_results,
            'searchType': 'words',
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()

        results = r.json().get('result', {}).get('results', [])
        for x in results:
            ui = (x.get('ui') or '').strip()
            if re.fullmatch(r'C\d+', ui):
                return ui, x.get('name', '')
        return None
    except Exception:
        return None


def _umls_get_semantic_types(cui, api_key):
    """Get semantic type codes (TUI) + names; includes fallback endpoint."""
    import re

    def _extract(st_list):
        tuis, names = [], []
        for st in st_list or []:
            if isinstance(st, dict):
                code = (
                    st.get('uri')
                    or st.get('tui')
                    or st.get('ui')
                    or st.get('code')
                    or st.get('name')
                )
                nm = st.get('name', '')
            else:
                code = str(st)
                nm = ''
            code = str(code)
            m = re.search(r'T\d{3}', code)
            if m:
                tuis.append(m.group(0))
                names.append(nm)
        return tuis, names

    try:
        # Primary endpoint
        r = requests.get(
            f"{UMLS_BASE}/content/current/CUI/{cui}",
            params={'apiKey': api_key},
            timeout=30,
        )
        r.raise_for_status()
        tuis, names = _extract(r.json().get('result', {}).get('semanticTypes', []))
        if tuis or names:
            return tuis, names

        # Fallback endpoint
        r2 = requests.get(
            f"{UMLS_BASE}/content/current/CUI/{cui}/semantictypes",
            params={'apiKey': api_key},
            timeout=30,
        )
        r2.raise_for_status()
        res = r2.json().get('result', [])
        st_list = res.get('semanticTypes', []) if isinstance(res, dict) else res
        return _extract(st_list)
    except Exception:
        return [], []

In [33]:
umls_rows = []
for name in tqdm(kg_nonmatched_names, desc='UMLS normalization'):
    search_term = _norm_name(name)
    original_entity_types =  kg_nonmatched_entities.get(name)
    best = _umls_search_best(search_term, UMLS_API_KEY)
    if not best:
        umls_rows.append({
            'entity_name': name,
            'original_entity_types': original_entity_types,
            'search_term_used': search_term,
            'umls_matched_name': None,
            'matched_cui': None,
            'matched_semantic_types': [],
            'matched_semantic_type_names': []
        })
        continue
    cui, umls_name = best
    st, st_names = _umls_get_semantic_types(cui, UMLS_API_KEY)
    umls_rows.append({
        'entity_name': name,
        'original_entity_types': original_entity_types,
        'search_term_used': search_term,
        'umls_matched_name': umls_name,
        'matched_cui': cui,
        'matched_semantic_types': st,
        'matched_semantic_type_names': st_names
    })

entity_umls_df = pd.DataFrame(umls_rows)


UMLS normalization: 100%|███████████████████████████████████████████████████████████████████████████████████████| 222/222 [02:35<00:00,  1.43it/s]


In [34]:
# Add PMID(s), relation(s), and entity_side(s) for each entity in entity_umls_df
name_to_pmids = {}
name_to_relations = {}
name_to_entity_sides = {}

for _, r in novel_relationships_df.iterrows():
    pmid = str(r.get("PMID") or "").strip()
    relation = str(r.get("Relation") or "").strip()

    n1 = str(r.get("entity1") or "").strip()
    if n1 and n1.lower() != "nan":
        if pmid and pmid.lower() != "nan":
            name_to_pmids.setdefault(n1, set()).add(pmid)
        if relation and relation.lower() != "nan":
            name_to_relations.setdefault(n1, set()).add(relation)
        name_to_entity_sides.setdefault(n1, set()).add("1")

    n2 = str(r.get("entity2") or "").strip()
    if n2 and n2.lower() != "nan":
        if pmid and pmid.lower() != "nan":
            name_to_pmids.setdefault(n2, set()).add(pmid)
        if relation and relation.lower() != "nan":
            name_to_relations.setdefault(n2, set()).add(relation)
        name_to_entity_sides.setdefault(n2, set()).add("2")

entity_umls_df["PMID"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_pmids.get(str(x).strip(), set())))
)
entity_umls_df["relation"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_relations.get(str(x).strip(), set())))
)
entity_umls_df["entity_side"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_entity_sides.get(str(x).strip(), set())))
)
# Cleanup typo column from previous runs if it exists.
if "entitiy_side" in entity_umls_df.columns:
    entity_umls_df = entity_umls_df.drop(columns=["entitiy_side"])
entity_umls_df.head(5)

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side
0,aav gene therapy,,aav gene therapy,NaN,NaN,[],[],,,
1,aav-hex gene therapy and bone marrow transplan...,,aav-hex gene therapy and bone marrow transplan...,NaN,NaN,[],[],,,
2,aav-php.eb carrying the abe,,aav-php.eb carrying the abe,NaN,NaN,[],[],,,
3,aav9-gm2a gene therapy,,aav9-gm2a gene therapy,NaN,NaN,[],[],,,
4,ab-variant gm2 gangliosidosis,,ab-variant gm2 gangliosidosis,"Tay-Sachs Disease, AB Variant",C0268275,[T047],[Disease or Syndrome],,,


In [35]:
#dont change code above this line
entity_umls_df_bkup = entity_umls_df.copy()
entity_umls_df = entity_umls_df_bkup
entity_umls_df.head(10)

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side
0,aav gene therapy,,aav gene therapy,NaN,NaN,[],[],,,
1,aav-hex gene therapy and bone marrow transplan...,,aav-hex gene therapy and bone marrow transplan...,NaN,NaN,[],[],,,
2,aav-php.eb carrying the abe,,aav-php.eb carrying the abe,NaN,NaN,[],[],,,
3,aav9-gm2a gene therapy,,aav9-gm2a gene therapy,NaN,NaN,[],[],,,
4,ab-variant gm2 gangliosidosis,,ab-variant gm2 gangliosidosis,"Tay-Sachs Disease, AB Variant",C0268275,[T047],[Disease or Syndrome],,,
5,abcc1 transporter,,abcc1 transporter,Multidrug Resistance Associated Protein 1,C0906368,"[T116, T123]","[Amino Acid, Peptide, or Protein, Biologically...",,,
6,abnormally elevated ca2+ influx,,abnormally elevated ca2+ influx,NaN,NaN,[],[],,,
7,accelerated hexa release,,accelerated hexa release,NaN,NaN,[],[],,,
8,accelerated mature cathepsins release,phenotype,accelerated mature cathepsins release,NaN,NaN,[],[],10.3390/cells10113122,disease_phenotype_positive,2
9,activated glial suppression,phenotype,activated glial suppression,NaN,NaN,[],[],10.3390/cells12242791,drug_effect,2


In [36]:
# Add semantic-type quality flag: out_of_expected_tuis (only meaningful when CUI exists)
import ast

def _to_list_safe(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
    return [s]

#test
# Case 1: Already a list — keep as-is
print(_to_list_safe(["T047", "T048"]))         # ["T047", "T048"]

# Case 2: None → []
print(_to_list_safe(None))                      # []

# Case 3: NaN → []
print(_to_list_safe(float("nan")))              # []
print(_to_list_safe(pd.NA))                     # []

# Case 4: Empty string → []
print(_to_list_safe("   "))                     # []

# Case 5: Valid list string → parse to list
print(_to_list_safe("['T047', 'T048']"))        # ["T047", "T048"]
print(_to_list_safe("['T047']"))                # ["T047"]

# Case 6: Invalid list string → wrap as [s]
print(_to_list_safe("[abc, def]"))              # ["[abc, def]"]

# Case 7: Plain string → wrap as list
print(_to_list_safe("T047"))                    # ["T047"]
print(_to_list_safe("  Canavan disease  "))     # ["Canavan disease"]

['T047', 'T048']
[]
[]
['<NA>']
[]
['T047', 'T048']
['T047']
['[abc, def]']
['T047']
['Canavan disease']


In [37]:
import ast

def _to_list_safe(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
    return [s]


def _expected_tuis_from_entity_types(entity_types):
    exp = set()
    for et in _to_list_safe(entity_types):
        key = str(et).strip().lower()
        for tui in ENTITY_TYPE_TO_TUI.get(key, []):
            tui_clean = str(tui).strip().upper()
            if tui_clean:
                exp.add(tui_clean)
    return exp


def _observed_tuis(semantic_types_col):
    obs = set()
    for t in _to_list_safe(semantic_types_col):
        tv = str(t).strip().upper()
        if tv:
            obs.add(tv)
    return obs



In [38]:
for t in ["disease", "phenotype", "drug"]:
    print(_expected_tuis_from_entity_types(t))

{'T191', 'T047', 'T048'}
{'T041', 'T184', 'T019', 'T034', 'T033', 'T020'}
{'T195', 'T121', 'T200'}


In [39]:
print(len(entity_umls_df))
entity_umls_df = entity_umls_df[entity_umls_df.original_entity_types!=""]
print(len(entity_umls_df))


222
104


In [40]:
entity_with_cui_after_first_search = entity_umls_df[~entity_umls_df.matched_cui.isna()]
len(entity_with_cui_after_first_search)

53

In [41]:
entity_umls_df.columns

Index(['entity_name', 'original_entity_types', 'search_term_used',
       'umls_matched_name', 'matched_cui', 'matched_semantic_types',
       'matched_semantic_type_names', 'PMID', 'relation', 'entity_side'],
      dtype='str')

In [42]:
entity_without_cui_after_first_search = entity_umls_df[entity_umls_df.matched_cui.isna()]
entity_without_cui_after_first_search = entity_without_cui_after_first_search[['entity_name', 'original_entity_types', "PMID", 'relation', 'entity_side']]
entity_without_cui_after_first_search

,entity_name,original_entity_types,PMID,relation,entity_side
8,accelerated mature cathepsins release,phenotype,10.3390/cells10113122,disease_phenotype_positive,2
9,activated glial suppression,phenotype,10.3390/cells12242791,drug_effect,2
13,ameliorated late-stage disease severity,phenotype,10.1371/journal.pone.0315005,drug_effect,2
14,amyloid-beta accumulation,phenotype,10.1016/j.ibneur.2022.01.004,disease_phenotype_positive|phenotype_protein,2
16,attenuated glial activation,phenotype,10.1515/nipt-2023-0027,drug_effect,2
41,cerebellar symptoms,phenotype,10.5334/tohm.726,disease_phenotype_positive,2
43,chronic neuroinflammation,phenotype,10.1007/s11011-025-01553-6,disease_phenotype_positive,2
46,decreased phosphatidylcholine,cellular_component,10.3389/fmolb.2022.892248,cellcomp_protein,2
47,decreased phosphatidylserine,cellular_component,10.3389/fmolb.2022.892248,cellcomp_protein,2
56,elevated phosphatidylcholine-ether,cellular_component,10.3389/fmolb.2022.892248,cellcomp_protein,2


In [90]:
entity_without_cui_after_first_search.to_csv(str(POST_DIR / "20260521-Tay-Sachs_disease_For_term_suggestion_before_BERT.csv"))

In [45]:
out_flags = []
expected_tuis_col = []

for _, r in entity_with_cui_after_first_search.iterrows(): #only apply for entity that has cui after searching
    cui = r.get("matched_cui")
    has_cui = not (
        cui is None
        or (isinstance(cui, float) and pd.isna(cui))
        or str(cui).strip() == ""
    )
    print("*** \n ",cui, " has cui ", has_cui)
    
    expected_tuis = _expected_tuis_from_entity_types(r.get("original_entity_types", []))
    observed_tuis = _observed_tuis(r.get("matched_semantic_types", []))  # use correct column
    
    expected_tuis_col.append(sorted(expected_tuis))
    
    # out_of_expected_tuis = True khi:
    # - has CUI (comparison meaningful)
    # - has both expected and observed
    # - no overlap between observed and expected
    print("expected_tuis: ", expected_tuis, "observed_tuis: ", observed_tuis), 
    out_of_expected = bool(
        has_cui
        and bool(expected_tuis)
        and bool(observed_tuis)
        and len(expected_tuis & observed_tuis) == 0  # ← changed from > 0 to == 0
    )
    print(len(expected_tuis & observed_tuis), len(expected_tuis & observed_tuis)==0)
    print("out_of_expected: ", out_of_expected)
    out_flags.append(out_of_expected)

entity_with_cui_after_first_search["expected_tuis"] = expected_tuis_col
entity_with_cui_after_first_search["out_of_expected_tuis"] = pd.Series(
    out_flags,
    index=entity_with_cui_after_first_search.index,
    dtype=bool,
)
# Defensive cleanup in case this cell is re-run after earlier buggy assignments.
entity_with_cui_after_first_search["out_of_expected_tuis"] = (
    entity_with_cui_after_first_search["out_of_expected_tuis"]
    .fillna(False)
    .astype(bool)
)

*** 
  C1155612  has cui  True
expected_tuis:  {'T043'} observed_tuis:  {'T043'}
1 False
out_of_expected:  False
*** 
  C1818529  has cui  True
expected_tuis:  {'T021', 'T024', 'T018', 'T029', 'T017', 'T030', 'T022', 'T023'} observed_tuis:  {'T043'}
0 True
out_of_expected:  True
*** 
  C0278234  has cui  True
expected_tuis:  {'T041', 'T184', 'T019', 'T034', 'T033', 'T020'} observed_tuis:  {'T184'}
1 False
out_of_expected:  False
*** 
  C0011304  has cui  True
expected_tuis:  {'T041', 'T184', 'T019', 'T034', 'T033', 'T020'} observed_tuis:  {'T046'}
0 True
out_of_expected:  True
*** 
  C0424605  has cui  True
expected_tuis:  {'T041', 'T184', 'T019', 'T034', 'T033', 'T020'} observed_tuis:  {'T048'}
0 True
out_of_expected:  True
*** 
  C3494263  has cui  True
expected_tuis:  {'T041', 'T184', 'T019', 'T034', 'T033', 'T020'} observed_tuis:  {'T201'}
0 True
out_of_expected:  True
*** 
  C1271991  has cui  True
expected_tuis:  {'T041', 'T184', 'T019', 'T034', 'T033', 'T020'} observed_tuis:  {'

In [46]:
entity_with_cui_after_first_search

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side,expected_tuis,out_of_expected_tuis
17,autophagy inducer,molecular_function,autophagy inducer,regulation of macroautophagy,C1155612,[T043],[Cell Function],10.1016/j.ymgme.2024.108140,drug_bioprocess,2,[T043],False
40,central nervous system neurons,anatomy,central nervous system neurons,central nervous system neuron development,C1818529,[T043],[Cell Function],10.1016/j.ymgme.2021.05.001,anatomy_disease,2,"[T017, T018, T021, T022, T023, T024, T029, T030]",True
42,cherry-red spot,phenotype,cherry-red spot,Cherry red spot,C0278234,[T184],[Sign or Symptom],10.1016/j.ajoc.2025.102381|10.5546/aap.2022.en...,disease_phenotype_positive|phenotype_phenotype,2,"[T019, T020, T033, T034, T041, T184]",False
49,demyelination,phenotype,demyelination,Demyelination,C0011304,[T046],[Pathologic Function],10.3389/fmolb.2022.892248,disease_phenotype_positive,2,"[T019, T020, T033, T034, T041, T184]",True
50,developmental delay,phenotype,developmental delay,Developmental delay,C0424605,[T048],[Mental or Behavioral Dysfunction],10.1016/j.gim.2022.09.001|10.7759/cureus.51797,disease_phenotype_positive,2,"[T019, T020, T033, T034, T041, T184]",True
51,disease severity,phenotype,disease severity,Patient Acuity,C3494263,[T201],[Clinical Attribute],10.1016/j.ymgme.2022.106983,phenotype_phenotype,2,"[T019, T020, T033, T034, T041, T184]",True
53,early death,phenotype,early death,Early neonatal death,C1271991,[T033],[Finding],10.1016/j.ymgme.2024.108615|10.3390/jpm13081222,disease_phenotype_positive|phenotype_phenotype,2,"[T019, T020, T033, T034, T041, T184]",False
54,elevated levels of proinflammatory cytokines,phenotype,elevated levels of proinflammatory cytokines,Elevated proinflammatory cytokine levels,C5194194,[T033],[Finding],10.1007/s12031-025-02395-8,disease_phenotype_positive,2,"[T019, T020, T033, T034, T041, T184]",False
55,elevated lysophosphatidylcholine,cellular_component,elevated lysophosphatidylcholine,Elevated circulating lysophosphatidylcholine c...,C5937189,[T033],[Finding],10.3389/fmolb.2022.892248,cellcomp_protein,2,[],False
67,fall,phenotype,fall,Falls,C0085639,[T033],[Finding],10.1016/j.gim.2025.101615,drug_effect,2,"[T019, T020, T033, T034, T041, T184]",False


In [47]:
# Threshold: entity is considered "in KG" by SapBERT if max cosine sim with KG >= threshold
SAPBERT_IN_KG_THRESHOLD = 0.5
# Top-k candidates from SapBERT to rerank with SBERT
TOP_K_SAPBERT = 20

In [48]:
# --- SapBERT: load model and define embed/cosine helpers ---
# Input: SAPBERT_MODEL, DEVICE (from config).
# Expected output: tokenizer, sapbert_model; functions embed_names, cos_sim.

def load_sapbert():
    tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
    model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE)
    return tokenizer, model

def embed_names(tokenizer, model, names, batch_size=None):
    if not names:
        return np.zeros((0, 768))
    batch_size = batch_size or SAPBERT_BATCH
    names = [str(n).strip() or " " for n in names]
    embs = []
    for i in range(0, len(names), batch_size):
        batch = names[i : i + batch_size]
        toks = tokenizer(batch, padding="max_length", max_length=SAPBERT_MAX_LEN, truncation=True, return_tensors="pt")
        toks = {k: v.to(DEVICE) for k, v in toks.items()}
        with torch.no_grad():
            cls_rep = model(**toks)[0][:, 0, :]
        embs.append(cls_rep.cpu().numpy())
    return np.concatenate(embs, axis=0)

def cos_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))

tokenizer, sapbert_model = load_sapbert()

Loading weights: 100%|███████████████████████████████████████████████| 199/199 [00:00<00:00, 1775.65it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
def load_sbert():
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    return model

def embed_names_sbert(model, names):
    return model.encode(
        names,
        convert_to_numpy=True,
        normalize_embeddings=True  # important for cosine via dot product
    )

sbert_model = load_sbert()

Loading weights: 100%|███████████████████████████████████████████████| 103/103 [00:00<00:00, 1534.36it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
# Cell 107: SapBERT top-20 nearest PrimeKG names, then SBERT rerank
# Requires: novel_relationships_df (with entity*_cui, entity*_tui_out_of_expected_group),
#           tokenizer, sapbert_model, embed_names, DEVICE, SAPBERT_BATCH,
#           sbert_model, embed_names_sbert, KG_FILE
TOP_K = 20

def load_primekg_display_names(kg_path: Path) -> list[str]:
    kg = pd.read_csv(kg_path, low_memory=False)
    name_cols = [c for c in kg.columns if c.endswith("_name")]
    if not name_cols:
        raise ValueError(f"No columns ending with _name in {kg_path}")
    acc: list[str] = []
    for c in name_cols:
        acc.extend(kg[c].dropna().astype(str).str.strip().tolist())
    return sorted({x for x in acc if x and x.lower() != "nan"})


def _l2norm_rows(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-9)

def _normalize_curated_pmid(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return ""
    if s.endswith(".0") and s[:-2].replace(".", "", 1).isdigit():
        s = s[:-2]
    return s


primekg_entity_names_list = load_primekg_display_names(KG_FILE)

In [51]:
len(primekg_entity_names_list)

128278

In [52]:
len(entity_with_cui_after_first_search), len(entity_without_cui_after_first_search)

(53, 51)

In [53]:
len(entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis])

30

In [54]:
entity_with_cui_after_first_search.columns, entity_without_cui_after_first_search.columns

(Index(['entity_name', 'original_entity_types', 'search_term_used',
        'umls_matched_name', 'matched_cui', 'matched_semantic_types',
        'matched_semantic_type_names', 'PMID', 'relation', 'entity_side',
        'expected_tuis', 'out_of_expected_tuis'],
       dtype='str'),
 Index(['entity_name', 'original_entity_types', 'PMID', 'relation',
        'entity_side'],
       dtype='str'))

In [55]:
query_names2 = list(entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis].entity_name.unique()) + list(entity_without_cui_after_first_search.entity_name.unique())
query_names2 = list(set(query_names2))
# primekg_entity_names_list = load_primekg_display_names(KG_FILE)
len(query_names2)

81

In [46]:
# --- SapBERT: embed all KG names and all query names ---
#time consuming, run once
# kg_sapbert_embeddings = embed_names(tokenizer, sapbert_model, primekg_entity_names_list)

In [58]:
query_sapbert_embeddings = embed_names(tokenizer, sapbert_model, query_names2)
query_sbert_embeddings = embed_names_sbert(sbert_model, query_names2)

In [59]:
#load heavy embeddings from files
EMBED_DIR = "./embeddings_cache"

kg_sapbert_embeddings = np.load(f"{EMBED_DIR}/kg_sapbert_embeddings.npy")
kg_sbert_embeddings   = np.load(f"{EMBED_DIR}/kg_sbert_embeddings.npy")

with open(f"{EMBED_DIR}/primekg_entity_names_list.txt") as f:
    primekg_entity_names_list = f.read().splitlines()

print(f"Loaded SapBERT: {kg_sapbert_embeddings.shape}")
print(f"Loaded SBERT:   {kg_sbert_embeddings.shape}")

Loaded SapBERT: (128278, 768)
Loaded SBERT:   (128278, 384)


In [60]:
kg_sap_n = _l2norm_rows(kg_sapbert_embeddings)
q_sap_n = _l2norm_rows(query_sapbert_embeddings)


k = min(TOP_K, len(primekg_entity_names_list))

In [61]:
import os
# os.makedirs(EMBED_DIR, exist_ok=True)

# np.save(f"{EMBED_DIR}/kg_sapbert_embeddings.npy", kg_sapbert_embeddings)
# np.save(f"{EMBED_DIR}/kg_sbert_embeddings.npy",   kg_sbert_embeddings)

# # Save name list for later verification
# with open(f"{EMBED_DIR}/primekg_entity_names_list.txt", "w") as f:
#     f.write("\n".join(primekg_entity_names_list))

# print(f"Saved {kg_sapbert_embeddings.shape} SapBERT embeddings")
# print(f"Saved {kg_sbert_embeddings.shape} SBERT embeddings")

In [62]:
rows: list[dict] = []

for i, orig in enumerate(query_names2):
    sims = kg_sap_n @ q_sap_n[i]  # (n_kg,)
    if k >= len(sims):
        top_idx = np.argsort(-sims)
    else:
        # argpartition for top-k without full sort
        part = np.argpartition(-sims, k - 1)[:k]
        top_idx = part[np.argsort(-sims[part])]

    top_idx = top_idx[:k]
    qvec = query_sbert_embeddings[i : i + 1]  # (1, dim)
    sbert_scores = np.dot(kg_sbert_embeddings[top_idx], qvec.T).flatten()
    best_rel = int(np.argmax(sbert_scores))
    best_kg_idx = int(top_idx[best_rel])
    suggested = primekg_entity_names_list[best_kg_idx]
    rows.append({"original_name": orig, "suggested_name": suggested})

suggested_name_replacement_df = pd.DataFrame(rows, columns=["original_name", "suggested_name"])

# Enrich with metadata columns requested by downstream export.
extra_cols = ["entity_name", "original_entity_types", "PMID", "relation", "entity_side"]
meta_df = entity_umls_df[extra_cols].drop_duplicates(subset=["entity_name"], keep="first")
meta_df = meta_df.rename(columns={"entity_name": "original_name"})
suggested_name_replacement_df = suggested_name_replacement_df.merge(
    meta_df,
    on="original_name",
    how="left",
)

suggested_name_replacement_df = suggested_name_replacement_df[[
    "original_name",
    "suggested_name",
    "original_entity_types",
    "PMID",
    "relation",
    "entity_side",
]]

suggested_name_replacement_df.head(15)

,original_name,suggested_name,original_entity_types,PMID,relation,entity_side
0,partial exon skipping,exon-exon junction complex disassembly,biological_process,10.1093/qjmed/hcaf246,bioprocess_protein,2
1,normalized neurodegenerative phenotype,inherited neurodegenerative disorder,phenotype,10.1016/j.omtm.2022.03.011,drug_effect,2
2,increased lyso-platelet activating factor,Increased level of platelet-activating factor,phenotype,10.1016/j.ymgme.2024.108615,phenotype_phenotype,2
3,reduced inflammatory cytokines,negative regulation of cytokine production inv...,phenotype,10.3390/life11101007,drug_effect,2
4,protein translation readthrough of stop codons,regulation of translational termination,biological_process,10.1038/s41586-025-09732-2,drug_bioprocess,2
5,elevated levels of proinflammatory cytokines,Abnormality of serum cytokine level,phenotype,10.1007/s12031-025-02395-8,disease_phenotype_positive,2
6,decreased phosphatidylcholine,negative regulation of phosphatidylcholine cat...,cellular_component,10.3389/fmolb.2022.892248,cellcomp_protein,2
7,perifoveal whitening,Abnormality of foveal pigmentation,phenotype,10.1016/j.ajoc.2025.102381,phenotype_anatomy,2
8,reduced lysosomal mass,lysosomal matrix,phenotype,10.3390/pharmaceutics17050628,drug_effect,2
9,hippocampus,hippocampal field,anatomy,10.3389/fmolb.2022.892248,anatomy_protein_present,1


In [63]:
len(suggested_name_replacement_df)

81

In [64]:
ENTITY_TYPE_ALIASES = {"protein/gene": "gene/protein"}

def _etype_to_expected_tuis(etype):
    if pd.isna(etype) or etype == "":
        return set()
    key = str(etype).strip().lower()
    key = ENTITY_TYPE_ALIASES.get(key, key)
    return set(tui.upper() for tui in (ENTITY_TYPE_TO_TUI.get(key, []) or []))


In [65]:
def _expected_type_label_and_tuis_for_original_name(orig: str) -> tuple[str, set[str]]:
    """From suggestion metadata for original name, build union of expected TUIs."""
    o = str(orig).strip()
    matched_types = suggested_name_replacement_df.loc[
        suggested_name_replacement_df["original_name"] == o,
        "original_entity_types",
    ].dropna()
    if matched_types.empty:
        return "", set()

    raw = str(matched_types.iloc[0]).strip()
    labels = sorted({x.strip() for x in raw.split("|") if x.strip()})
    if not labels:
        return "", set()
    exp_union: set[str] = set()
    for lab in labels:
        exp_union |= _etype_to_expected_tuis(lab)
    exp_union.discard("")  # skip empty TUI from ENTITY_TYPE_TO_TUI mapping
    return " | ".join(labels), exp_union


def build_suggested_umls_typecheck_df(suggestions: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict] = []
    for _, r in tqdm(suggestions.iterrows(), total=len(suggestions), desc="UMLS re-search suggested names"):
        orig = r["original_name"]
        sug = r["suggested_name"]
        pmid = r.get("PMID", pd.NA)
        relation = r.get("relation", pd.NA)
        entity_side = r.get("entity_side", pd.NA)
        exp_label, exp_tuis = _expected_type_label_and_tuis_for_original_name(orig)
        # print(orig, "|", sug, "|", exp_label, "|", exp_tuis )

        best = _umls_search_best(str(sug).strip(), UMLS_API_KEY)
        if not best:
            rows.append(
                {
                    "original_name": orig,
                    "suggested_name": sug,
                    "expected_entity_type": exp_label,
                    "expected_tuis": ",".join(sorted(exp_tuis)),
                    "PMID": pmid,
                    "relation": relation,
                    "entity_side": entity_side,
                    "umls_search_ok": False,
                    "suggested_cui": pd.NA,
                    "suggested_umls_name": pd.NA,
                    "suggested_semantic_types": [],
                    "suggested_semantic_type_names": [],
                    "type_match_expected_category": pd.NA,
                }
            )
            continue

        cui, umls_name = best
        st_list, st_names = _umls_get_semantic_types(cui, UMLS_API_KEY)
        found = {str(x).strip().upper() for x in (st_list or []) if x}

        if not exp_tuis:
            match = pd.NA
        elif not found:
            match = pd.NA
        else:
            match = bool(found & exp_tuis)

        rows.append(
            {
                "original_name": orig,
                "suggested_name": sug,
                "expected_entity_type": exp_label,
                "expected_tuis": ",".join(sorted(exp_tuis)),
                "PMID": pmid,
                "relation": relation,
                "entity_side": entity_side,
                "umls_search_ok": True,
                "suggested_cui": cui,
                "suggested_umls_name": umls_name,
                "suggested_semantic_types": list(st_list or []),
                "suggested_semantic_type_names": list(st_names or []),
                "type_match_expected_category": match,
            }
        )

    return pd.DataFrame(rows)


suggested_umls_typecheck_df = build_suggested_umls_typecheck_df(suggested_name_replacement_df)


UMLS re-search suggested names: 100%|█████████████████████████████████████████████████████████████████████████████| 81/81 [01:19<00:00,  1.02it/s]


In [66]:
suggested_umls_typecheck_df

,original_name,suggested_name,expected_entity_type,expected_tuis,PMID,relation,entity_side,umls_search_ok,suggested_cui,suggested_umls_name,suggested_semantic_types,suggested_semantic_type_names,type_match_expected_category
0,partial exon skipping,exon-exon junction complex disassembly,biological_process,"T038,T039",10.1093/qjmed/hcaf246,bioprocess_protein,2,True,C3894099,exon-exon junction complex disassembly,[T044],[Molecular Function],False
1,normalized neurodegenerative phenotype,inherited neurodegenerative disorder,phenotype,"T019,T020,T033,T034,T041,T184",10.1016/j.omtm.2022.03.011,drug_effect,2,False,NaN,NaN,[],[],<NA>
2,increased lyso-platelet activating factor,Increased level of platelet-activating factor,phenotype,"T019,T020,T033,T034,T041,T184",10.1016/j.ymgme.2024.108615,phenotype_phenotype,2,True,C4073143,Increased level of platelet-activating factor,[T033],[Finding],True
3,reduced inflammatory cytokines,negative regulation of cytokine production inv...,phenotype,"T019,T020,T033,T034,T041,T184",10.3390/life11101007,drug_effect,2,True,C3269276,negative regulation of cytokine production inv...,[T040],[Organism Function],False
4,protein translation readthrough of stop codons,regulation of translational termination,biological_process,"T038,T039",10.1038/s41586-025-09732-2,drug_bioprocess,2,True,C1157545,regulation of translational termination,[T043],[Cell Function],False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,elevated lysophosphatidylcholine,Elevated circulating lysophosphatidylcholine c...,cellular_component,,10.3389/fmolb.2022.892248,cellcomp_protein,2,True,C5937189,Elevated circulating lysophosphatidylcholine c...,[T033],[Finding],<NA>
77,enzyme β-hexosaminidase deficiency,Increased serum beta-hexosaminidase,phenotype,"T019,T020,T033,T034,T041,T184",10.3390/ijms22136751,disease_phenotype_positive,2,True,C2673361,Increased serum beta-hexosaminidase,[T033],[Finding],True
78,iminosugar compounds,Aminophosphonic acid-guanylate ester,drug,"T121,T195,T200",10.1080/14756366.2022.2073444,drug_effect,1,False,NaN,NaN,[],[],<NA>
79,extended lifspan,extension,phenotype,"T019,T020,T033,T034,T041,T184",10.1172/JCI183434,drug_effect,2,True,C0231448,Extension,[T169],[Functional Concept],False


In [89]:
suggested_umls_typecheck_df.to_csv(str(POST_DIR / "20260521-Tay-Sachs_suggested_terms_after_second_UMLS_search.csv"))
suggested_umls_typecheck_df = pd.read_csv(str(POST_DIR / "20260521-Tay-Sachs_suggested_terms_after_second_UMLS_search.csv"))

## Name-Filtering Pipeline and Stage Variables

This notebook filters curated entity names through PrimeKG and UMLS, then maps validated names back to relationship rows.

1) **Load and normalize curated relationships**
- `curated_relationships_df`: raw curated CSV input.
- `novel_relationships_df`: normalized relationship rows with `entity1`, `entity2`, `entity_type1`, `entity_type2`, `Relation`, and `PMID`.

2) **Build the initial query-name pool**
- `query_names`: unique names collected from both relationship sides.

3) **Run exact matching against PrimeKG**
- `kg_matched_names`: names that exact-match PrimeKG after normalization.
- `kg_nonmatched_names`: names that do not exact-match and require search-based handling.

4) **Attach curated entity-type context**
- `name_to_entity_types`: `name -> set(entity_type)` aggregated from both sides.
- `kg_matched_entities`, `kg_nonmatched_entities`: type context per bucket.

5) **Run UMLS normalization for nonmatched names**
- `entity_umls_df`: UMLS search output for nonmatched names (`matched_cui`, semantic types, etc.).
- Enriched metadata columns: `PMID`, `relation`, `entity_side`.

6) **Split by first-search CUI availability**
- `entity_with_cui_after_first_search`
- `entity_without_cui_after_first_search`

7) **Run semantic-type quality check on first-search CUI rows**
- Adds `expected_tuis` and `out_of_expected_tuis`.

8) **Build second-round suggestion inputs**
- `query_names` (round 2): quality-passed CUI names plus no-CUI names.

9) **Generate PrimeKG suggestions (SapBERT retrieve + SBERT rerank)**
- `suggested_name_replacement_df`:
  - `original_name`, `suggested_name`
  - merged metadata from `entity_umls_df`: `original_entity_types`, `PMID`, `relation`, `entity_side`.

10) **Type-check suggested names in UMLS**
- `suggested_umls_typecheck_df`:
  - re-search UMLS on `suggested_name`
  - compare expected TUIs vs found TUIs
  - output decision column: `type_match_expected_category`
  - carries curated metadata (`PMID`, `relation`, `entity_side`).

---

### Stage-wise name containers
- Initial pool: `query_names` (round 1)
- Exact matched: `kg_matched_names`
- Exact nonmatched: `kg_nonmatched_names`
- Nonmatched with UMLS context: `entity_umls_df`
- With first-search CUI: `entity_with_cui_after_first_search`
- Without first-search CUI: `entity_without_cui_after_first_search`
- Suggestion input pool: `query_names` (round 2, after quality filtering)
- Suggestion table: `suggested_name_replacement_df`
- Suggestion + type-check table: `suggested_umls_typecheck_df`

In [67]:
len(kg_matched_names)

44

In [68]:
valid_names_after_first_search = entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis].entity_name.unique()
len(valid_names_after_first_search)

30

In [69]:
valid_names_after_second_search = suggested_umls_typecheck_df[suggested_umls_typecheck_df.type_match_expected_category==True].original_name.unique()
len(valid_names_after_second_search)

27

In [70]:
[x for x in kg_matched_names if x in valid_names_after_first_search]

[]

In [71]:
[x for x in kg_matched_names if x in valid_names_after_second_search]

[]

In [72]:
all_good_names = kg_matched_names + list(valid_names_after_first_search) + list(valid_names_after_second_search)
len(all_good_names)

101

In [73]:
# Put validated names back to relationship rows (entity1/entity2)
# Rules:
# - in_kg: name is in kg_matched_names
# - CUI: name is valid after first/second search
# - invalid: name is not in all_good_names

def _norm_name_for_status(x) -> str:
    return str(x).strip().lower()

kg_matched_norm = {_norm_name_for_status(x) for x in kg_matched_names}
valid_first_norm = {_norm_name_for_status(x) for x in valid_names_after_first_search}
valid_second_norm = {_norm_name_for_status(x) for x in valid_names_after_second_search}
all_good_norm = {_norm_name_for_status(x) for x in all_good_names}


def _status_from_name(name: str) -> str:
    n = _norm_name_for_status(name)
    if not n or n == "nan":
        return "invalid"
    if n in kg_matched_norm:
        return "in_kg"
    if n in valid_first_norm or n in valid_second_norm:
        return "CUI"
    return "invalid"

# Keep original relationship table, and add statuses in a derived table
relationships_with_status_df = novel_relationships_df.copy()
relationships_with_status_df["entity1_status"] = relationships_with_status_df["entity1"].map(_status_from_name)
relationships_with_status_df["entity2_status"] = relationships_with_status_df["entity2"].map(_status_from_name)

# Optional quick checks
print("entity1_status counts:")
print(relationships_with_status_df["entity1_status"].value_counts(dropna=False))
print("\nentity2_status counts:")
print(relationships_with_status_df["entity2_status"].value_counts(dropna=False))

# relationships_with_status_df.head(10)

entity1_status counts:
entity1_status
invalid    168
in_kg      158
CUI          8
Name: count, dtype: int64

entity2_status counts:
entity2_status
invalid    190
in_kg       94
CUI         50
Name: count, dtype: int64


In [74]:
relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status
18,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1016/j.ymgme.2021.05.001,in_kg,in_kg
22,Tay-Sachs disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,CUI
23,Sandhoff disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,CUI
28,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1002/prot.26180,in_kg,in_kg
32,glioblastoma multiforme,HEXA,disease,protein/gene,disease_protein,10.3389/fonc.2021.685893,CUI,in_kg
...,...,...,...,...,...,...,...,...
320,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1016/j.gim.2025.101615,in_kg,in_kg
321,Sandhoff disease,HEXB,disease,protein,disease_protein,10.1016/j.gim.2025.101615,in_kg,in_kg
323,venglustat,fall,drug,phenotype,drug_effect,10.1016/j.gim.2025.101615,in_kg,CUI
324,venglustat,headache,drug,phenotype,drug_effect,10.1016/j.gim.2025.101615,in_kg,in_kg


In [75]:
# Ensure there is no literal "CUI" left in status columns.
# If a concrete CUI cannot be resolved, fallback to "invalid".

def _norm_name_for_status(x) -> str:
    return str(x).strip().lower()

# Rebuild name -> CUI map (first + second search)
name_to_cui = {}

first_valid_df = entity_with_cui_after_first_search[
    entity_with_cui_after_first_search.out_of_expected_tuis == True
].copy()
first_valid_df = first_valid_df[~first_valid_df["matched_cui"].isna()]
for _, r in first_valid_df.iterrows():
    n = _norm_name_for_status(r.get("entity_name"))
    c = str(r.get("matched_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)

second_valid_df = suggested_umls_typecheck_df[
    suggested_umls_typecheck_df.type_match_expected_category == True
].copy()
second_valid_df = second_valid_df[~second_valid_df["suggested_cui"].isna()]
for _, r in second_valid_df.iterrows():
    n = _norm_name_for_status(r.get("original_name"))
    c = str(r.get("suggested_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)


def _resolve_cui_or_invalid(name: str) -> str:
    n = _norm_name_for_status(name)
    cuis = sorted(name_to_cui.get(n, set()))
    return "|".join(cuis) if cuis else "invalid"

# Force-replace any remaining literal CUI in status columns
m1 = relationships_with_status_df["entity1_status"].astype(str).str.strip().eq("CUI")
m2 = relationships_with_status_df["entity2_status"].astype(str).str.strip().eq("CUI")

relationships_with_status_df.loc[m1, "entity1_status"] = relationships_with_status_df.loc[m1, "entity1"].map(_resolve_cui_or_invalid)
relationships_with_status_df.loc[m2, "entity2_status"] = relationships_with_status_df.loc[m2, "entity2"].map(_resolve_cui_or_invalid)

print("Remaining literal 'CUI' in entity1_status:", int((relationships_with_status_df["entity1_status"] == "CUI").sum()))
print("Remaining literal 'CUI' in entity2_status:", int((relationships_with_status_df["entity2_status"] == "CUI").sum()))

relationships_with_status_df.head(10)

Remaining literal 'CUI' in entity1_status: 0
Remaining literal 'CUI' in entity2_status: 0


,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status
0,GM2 gangliosidoses,Tay-sachs disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6,invalid,in_kg
1,GM2 gangliosidoses,Sandhoff disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6,invalid,in_kg
2,GM1 gangliosidoses,progressive neurodegeneration,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,invalid
3,GM2 gangliosidoses,progressive neurodegeneration,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,invalid
4,GM2 gangliosidoses,speech difficulties,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C1865313
5,GM2 gangliosidoses,mobility impairment,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,invalid
6,GM1 gangliosidoses,speech difficulties,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C1865313
7,GM1 gangliosidoses,mobility impairment,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,invalid
8,GM2 gangliosidoses,Tay-Sachs disease,disease,disease,disease_disease,10.1016/j.ymgme.2021.05.001,invalid,in_kg
9,Lysosomal storage disorders,GM2 gangliosidoses,disease,disease,disease_disease,10.1016/j.ymgme.2021.05.001,invalid,invalid


In [68]:
# relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

In [76]:
relationships_with_status_df[(relationships_with_status_df.entity1_status=="in_kg")&(relationships_with_status_df.entity2_status=="in_kg")]

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status
18,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1016/j.ymgme.2021.05.001,in_kg,in_kg
28,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1002/prot.26180,in_kg,in_kg
36,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1007/s12031-021-01907-6,in_kg,in_kg
37,Sandhoff disease,HEXB,disease,protein/gene,disease_protein,10.1007/s12031-021-01907-6,in_kg,in_kg
39,Metachromatic leukodystrophy,ARSA,disease,protein/gene,disease_protein,10.1007/s12031-021-01907-6,in_kg,in_kg
44,lysosomal storage disease,sphingolipidosis,disease,disease,disease_disease,10.1007/s12031-021-01907-6,in_kg,in_kg
45,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.3390/life11101007,in_kg,in_kg
50,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1007/s10072-021-05757-3,in_kg,in_kg
64,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.5546/aap.2022.eng.e25,in_kg,in_kg
65,Tay-Sachs disease,neurodegeneration,disease,phenotype,disease_phenotype_positive,10.5546/aap.2022.eng.e25,in_kg,in_kg


In [77]:
relationships_with_status_df[~((relationships_with_status_df.entity1_status=="in_kg")&(relationships_with_status_df.entity2_status=="in_kg")) &((relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid"))]

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status
22,Tay-Sachs disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,C2673361
23,Sandhoff disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,C2673361
52,Tay-Sachs disease,lower motor neuron disease,disease,disease,disease_disease,10.1007/s10072-021-05757-3,in_kg,C0085084
68,Tay-Sachs disease,cherry-red spot,disease,phenotype,disease_phenotype_positive,10.5546/aap.2022.eng.e25,in_kg,C2216370
104,gangliosidoses,macrocephaly,disease,phenotype,disease_phenotype_positive,10.1016/j.gim.2022.09.001,C0017083,in_kg
106,gangliosidoses,hypotonia,disease,phenotype,disease_phenotype_positive,10.1016/j.gim.2022.09.001,C0017083,in_kg
144,Tay-Sachs disease,metabolic disease,disease,Disease,disease_disease,10.3390/jpm13081222,in_kg,C0751744
179,lithium,autophagy inducer,drug,molecular_function,drug_bioprocess,10.1016/j.ymgme.2024.108140,in_kg,C5907175
200,rare genetic disease,Tay-Sachs disease,disease,disease,disease_disease,10.1186/s13023-024-03300-z,C0037277,in_kg
264,Tay-Sachs disease,elevated levels of proinflammatory cytokines,disease,phenotype,disease_phenotype_positive,10.1007/s12031-025-02395-8,in_kg,C4023535


## Final Consolidated Mapping Block

This block is the clean, production-ready version for mapping validated names back to relationship rows.

What it does:
- Builds `all_good_names` from exact KG matches + first-search valid names + second-search valid names.
- Builds one consolidated `name -> CUI` map from first and second UMLS searches.
- Creates `relationships_with_status_df` with:
  - `entity1_status`, `entity2_status`
  - values: `in_kg`, concrete CUI code(s) (`C...`), or `invalid`
- Adds second-search suggestion mapping:
  - `entity1_suggested_name`
  - `entity2_suggested_name`
  - `second_search_suggested_name` (combined view)

Notes:
- No debug `print` statements are used.
- Names not in `valid_names_after_second_search` get `None` in suggested-name columns.

In [78]:
# Consolidated final mapping (no debug prints)

def _norm_name(x) -> str:
    return str(x).strip().lower()

# 1) Final valid-name sets
valid_names_after_first_search = entity_with_cui_after_first_search[
    entity_with_cui_after_first_search["out_of_expected_tuis"] == False
]["entity_name"].dropna().astype(str).str.strip().unique()

valid_names_after_second_search = suggested_umls_typecheck_df[
    suggested_umls_typecheck_df["type_match_expected_category"] == True
]["original_name"].dropna().astype(str).str.strip().unique()

all_good_names = sorted(set(kg_matched_names) | set(valid_names_after_first_search) | set(valid_names_after_second_search))

# 2) Build one name -> CUI map from first and second search results
name_to_cui = {}

first_valid_df = entity_with_cui_after_first_search[
    (entity_with_cui_after_first_search["out_of_expected_tuis"] == False)
    & (~entity_with_cui_after_first_search["matched_cui"].isna())
]
for _, r in first_valid_df.iterrows():
    n = _norm_name(r.get("entity_name"))
    c = str(r.get("matched_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)

second_valid_df = suggested_umls_typecheck_df[
    (suggested_umls_typecheck_df["type_match_expected_category"] == True)
    & (~suggested_umls_typecheck_df["suggested_cui"].isna())
]
for _, r in second_valid_df.iterrows():
    n = _norm_name(r.get("original_name"))
    c = str(r.get("suggested_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)

kg_matched_norm = {_norm_name(x) for x in kg_matched_names}


def _status_from_name(name: str) -> str:
    n = _norm_name(name)
    if not n or n == "nan":
        return "invalid"
    if n in kg_matched_norm:
        return "in_kg"
    cuis = sorted(name_to_cui.get(n, set()))
    if cuis:
        return "|".join(cuis)
    return "invalid"

# 3) Build status output mapped back to curated relationships
relationships_with_status_df = novel_relationships_df.copy()
relationships_with_status_df["entity1_status"] = relationships_with_status_df["entity1"].map(_status_from_name)
relationships_with_status_df["entity2_status"] = relationships_with_status_df["entity2"].map(_status_from_name)

# 4) Add second-search suggested names for original names only
valid_second_norm = {_norm_name(x) for x in valid_names_after_second_search}
second_search_map_df = suggested_name_replacement_df[
    suggested_name_replacement_df["original_name"].astype(str).str.strip().str.lower().isin(valid_second_norm)
][["original_name", "suggested_name"]].drop_duplicates(subset=["original_name"], keep="first")

second_search_name_to_suggested = {
    _norm_name(r["original_name"]): r["suggested_name"]
    for _, r in second_search_map_df.iterrows()
}

relationships_with_status_df["entity1_suggested_name"] = relationships_with_status_df["entity1"].map(
    lambda x: second_search_name_to_suggested.get(_norm_name(x), None)
)
relationships_with_status_df["entity2_suggested_name"] = relationships_with_status_df["entity2"].map(
    lambda x: second_search_name_to_suggested.get(_norm_name(x), None)
)


def _combined_suggestion(row):
    s1 = row.get("entity1_suggested_name")
    s2 = row.get("entity2_suggested_name")
    if pd.notna(s1) and pd.notna(s2):
        return f"entity1:{s1}|entity2:{s2}"
    if pd.notna(s1):
        return s1
    if pd.notna(s2):
        return s2
    return None

relationships_with_status_df["second_search_suggested_name"] = relationships_with_status_df.apply(
    _combined_suggestion,
    axis=1,
)

relationships_with_status_df.head(12)

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status,entity1_suggested_name,entity2_suggested_name,second_search_suggested_name
0,GM2 gangliosidoses,Tay-sachs disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6,invalid,in_kg,NaN,NaN,NaN
1,GM2 gangliosidoses,Sandhoff disease,disease,disease,disease_disease,10.1186/s13023-025-04030-6,invalid,in_kg,NaN,NaN,NaN
2,GM1 gangliosidoses,progressive neurodegeneration,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C1854838,NaN,NaN,NaN
3,GM2 gangliosidoses,progressive neurodegeneration,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C1854838,NaN,NaN,NaN
4,GM2 gangliosidoses,speech difficulties,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C0233715|C1865313,NaN,Speech articulation difficulties,Speech articulation difficulties
5,GM2 gangliosidoses,mobility impairment,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C5777030,NaN,NaN,NaN
6,GM1 gangliosidoses,speech difficulties,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C0233715|C1865313,NaN,Speech articulation difficulties,Speech articulation difficulties
7,GM1 gangliosidoses,mobility impairment,disease,phenotype,disease_phenotype_positive,10.1186/s13023-025-04030-6,invalid,C5777030,NaN,NaN,NaN
8,GM2 gangliosidoses,Tay-Sachs disease,disease,disease,disease_disease,10.1016/j.ymgme.2021.05.001,invalid,in_kg,NaN,NaN,NaN
9,Lysosomal storage disorders,GM2 gangliosidoses,disease,disease,disease_disease,10.1016/j.ymgme.2021.05.001,invalid,invalid,NaN,NaN,NaN


In [79]:
len(relationships_with_status_df)

334

In [80]:
relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status,entity1_suggested_name,entity2_suggested_name,second_search_suggested_name
18,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1016/j.ymgme.2021.05.001,in_kg,in_kg,NaN,NaN,NaN
22,Tay-Sachs disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,C2673361,NaN,Increased serum beta-hexosaminidase,Increased serum beta-hexosaminidase
23,Sandhoff disease,enzyme β-hexosaminidase deficiency,disease,phenotype,disease_phenotype_positive,10.3390/ijms22136751,in_kg,C2673361,NaN,Increased serum beta-hexosaminidase,Increased serum beta-hexosaminidase
28,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1002/prot.26180,in_kg,in_kg,NaN,NaN,NaN
32,glioblastoma multiforme,HEXA,disease,protein/gene,disease_protein,10.3389/fonc.2021.685893,C1621958,in_kg,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
320,Tay-Sachs disease,HEXA,disease,protein/gene,disease_protein,10.1016/j.gim.2025.101615,in_kg,in_kg,NaN,NaN,NaN
321,Sandhoff disease,HEXB,disease,protein,disease_protein,10.1016/j.gim.2025.101615,in_kg,in_kg,NaN,NaN,NaN
323,venglustat,fall,drug,phenotype,drug_effect,10.1016/j.gim.2025.101615,in_kg,C0085639|C0850703,NaN,Frequent falls,Frequent falls
324,venglustat,headache,drug,phenotype,drug_effect,10.1016/j.gim.2025.101615,in_kg,in_kg,NaN,NaN,NaN


In [75]:
# relationships_with_status_df[~relationships_with_status_df.second_search_suggested_name.isna()][["entity1", "entity2", "entity1_status", "entity2_status", "second_search_suggested_name"]]

In [81]:
final = relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

In [82]:
final.entity_type1.unique()

<StringArray>
['disease', 'drug']
Length: 2, dtype: str

In [83]:
final.entity_type2.unique()

<StringArray>
[      'protein/gene',          'phenotype',            'disease',
 'biological process',            'Disease', 'molecular_function',
            'protein']
Length: 7, dtype: str

In [84]:
len(final)

67

In [85]:
final = final.dropna(subset=["entity_type1", "entity_type2"])
len(final)

67

In [86]:
#Ask Thuy --> can we replace "Cell therapy" by "Drug"?
final.entity_type1.unique()

<StringArray>
['disease', 'drug']
Length: 2, dtype: str

In [87]:
final.entity_type2.unique()

<StringArray>
[      'protein/gene',          'phenotype',            'disease',
 'biological process',            'Disease', 'molecular_function',
            'protein']
Length: 7, dtype: str

In [88]:
final.to_csv(str(POST_DIR / "20260521-Tay-Sachs_final.csv"))